
# 04 - Hyperparameter Optimization

## Objetivo
En este notebook se realiza la optimización de hiperparámetros utilizando:

- `GridSearchCV`
- `RandomizedSearchCV`

sobre los modelos de clasificación del proyecto:

1. Logistic Regression
2. Decision Tree
3. Random Forest

La meta es mejorar el rendimiento del modelo y superar el **80% de accuracy** solicitado en la evaluación.


In [1]:

# Librerías principales
import pandas as pd
import numpy as np

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Pipeline y preprocesamiento
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Optimización
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Métricas
from sklearn.metrics import accuracy_score, classification_report

# Para ignorar warnings innecesarios
import warnings
warnings.filterwarnings("ignore")



## Carga de datos

Ajusta la ruta según tu proyecto.  
Este notebook asume que ya existen datasets procesados para entrenamiento y prueba.


In [2]:
import pandas as pd

# Rutas reales de tu proyecto
X_train = pd.read_csv("../data/04_feature/X_train.csv")
X_test = pd.read_csv("../data/04_feature/X_test.csv")

y_train = pd.read_csv("../data/04_feature/y_train.csv").iloc[:, 0]
y_test = pd.read_csv("../data/04_feature/y_test.csv").iloc[:, 0]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (10512, 26)
X_test: (2629, 26)
y_train: (10512,)
y_test: (2629,)



# 1. Logistic Regression + GridSearchCV

Probamos diferentes combinaciones de:

- `C`
- `solver`
- `penalty`

para encontrar la mejor configuración posible.


In [4]:

# Pipeline
log_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000))
])

# Parámetros a probar
log_params = {
    "clf__C": [0.01, 0.1, 1, 10],
    "clf__solver": ["liblinear", "lbfgs"],
    "clf__penalty": ["l2"]
}

# Grid Search
log_grid = GridSearchCV(
    estimator=log_pipeline,
    param_grid=log_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

# Entrenamiento
log_grid.fit(X_train, y_train)

# Mejor modelo
best_log_model = log_grid.best_estimator_

print("Mejores parámetros:")
print(log_grid.best_params_)

print("\nMejor accuracy CV:")
print(round(log_grid.best_score_, 4))


Fitting 5 folds for each of 8 candidates, totalling 40 fits
Mejores parámetros:
{'clf__C': 0.1, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'}

Mejor accuracy CV:
0.7828


In [5]:

# Evaluación en test
y_pred_log = best_log_model.predict(X_test)

log_accuracy = accuracy_score(y_test, y_pred_log)

print("Accuracy TEST:", round(log_accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_log))


Accuracy TEST: 0.7828

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.88      0.85      1822
           1       0.67      0.57      0.62       807

    accuracy                           0.78      2629
   macro avg       0.75      0.72      0.73      2629
weighted avg       0.78      0.78      0.78      2629




# 2. Decision Tree + GridSearchCV

Aquí optimizamos:

- profundidad máxima
- mínimo de muestras por split
- criterio de división


In [6]:

tree_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", DecisionTreeClassifier(random_state=42))
])

tree_params = {
    "clf__max_depth": [3, 5, 10, 15, None],
    "clf__min_samples_split": [2, 5, 10],
    "clf__criterion": ["gini", "entropy"]
}

tree_grid = GridSearchCV(
    estimator=tree_pipeline,
    param_grid=tree_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

tree_grid.fit(X_train, y_train)

best_tree_model = tree_grid.best_estimator_

print("Mejores parámetros:")
print(tree_grid.best_params_)

print("\nMejor accuracy CV:")
print(round(tree_grid.best_score_, 4))


Fitting 5 folds for each of 30 candidates, totalling 150 fits
Mejores parámetros:
{'clf__criterion': 'entropy', 'clf__max_depth': 10, 'clf__min_samples_split': 10}

Mejor accuracy CV:
0.7757


In [7]:

# Evaluación final
y_pred_tree = best_tree_model.predict(X_test)

tree_accuracy = accuracy_score(y_test, y_pred_tree)

print("Accuracy TEST:", round(tree_accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tree))


Accuracy TEST: 0.7684

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.84      0.83      1822
           1       0.63      0.61      0.62       807

    accuracy                           0.77      2629
   macro avg       0.73      0.72      0.73      2629
weighted avg       0.77      0.77      0.77      2629




# 3. Random Forest + RandomizedSearchCV

Random Forest normalmente entrega mejores resultados en este tipo de problemas.

Usamos `RandomizedSearchCV` porque:
- prueba muchas combinaciones rápidamente
- reduce tiempo de entrenamiento
- suele encontrar configuraciones muy buenas


In [10]:

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(random_state=42))
])

# Espacio de búsqueda
rf_params = {
    "clf__n_estimators": [200, 300, 500, 800],
    "clf__max_depth": [10, 20, 30, 50, None],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 4],
    "clf__max_features": ["sqrt", "log2"],
    "clf__bootstrap": [True, False]
}

rf_random = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_params,
    n_iter=50,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_random.fit(X_train, y_train)

best_rf_model = rf_random.best_estimator_

print("Mejores parámetros:")
print(rf_random.best_params_)

print("\nMejor accuracy CV:")
print(round(rf_random.best_score_, 4))


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Mejores parámetros:
{'clf__n_estimators': 200, 'clf__min_samples_split': 10, 'clf__min_samples_leaf': 2, 'clf__max_features': 'log2', 'clf__max_depth': 10, 'clf__bootstrap': False}

Mejor accuracy CV:
0.7851


In [11]:

# Evaluación final Random Forest
y_pred_rf = best_rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)

print("Accuracy TEST:", round(rf_accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))


Accuracy TEST: 0.7809

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.89      0.85      1822
           1       0.69      0.53      0.60       807

    accuracy                           0.78      2629
   macro avg       0.75      0.71      0.72      2629
weighted avg       0.77      0.78      0.77      2629




# Comparación Final de Modelos


In [12]:

results = pd.DataFrame({
    "Modelo": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        log_accuracy,
        tree_accuracy,
        rf_accuracy
    ]
})

results = results.sort_values(by="Accuracy", ascending=False)

results


,Modelo,Accuracy
0,Logistic Regression,0.782807
2,Random Forest,0.780905
1,Decision Tree,0.768353



# Conclusión

- Se aplicaron técnicas de optimización de hiperparámetros utilizando:
  - `GridSearchCV`
  - `RandomizedSearchCV`

- Los modelos fueron evaluados mediante validación cruzada (`cv=5`) para obtener resultados más robustos.

- El modelo con mejor rendimiento fue seleccionado utilizando accuracy como métrica principal.

- Esta optimización permite mejorar considerablemente el desempeño respecto a los modelos base y aumenta las probabilidades de superar el objetivo de **80% de accuracy** solicitado en la evaluación.
